<a href="https://colab.research.google.com/github/iloveunk1310/AINoob_AIC25/blob/main/train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Installation

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install jiwer
!pip install einops addict easydict

In [2]:
from huggingface_hub import snapshot_download
snapshot_download("unsloth/DeepSeek-OCR", local_dir = "deepseek_ocr")
from unsloth import FastVisionModel # FastLanguageModel for LLMs
import torch
from transformers import AutoModel
import os
os.environ["UNSLOTH_WARN_UNINITIALIZED"] = '0'
# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Qwen3-VL-8B-Instruct-bnb-4bit", # Qwen 3 vision support
    "unsloth/Qwen3-VL-8B-Thinking-bnb-4bit",
    "unsloth/Qwen3-VL-32B-Instruct-bnb-4bit",
    "unsloth/Qwen3-VL-32B-Thinking-bnb-4bit",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastVisionModel.from_pretrained(
    "./deepseek_ocr",
    load_in_4bit = False, # Use 4bit to reduce memory use. False for 16bit LoRA.
    auto_model = AutoModel,
    trust_remote_code=True,
    unsloth_force_compile=True,
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for long context
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 21 files:   0%|          | 0/21 [00:00<?, ?it/s]

README-checkpoint.md: 0.00B [00:00, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

LICENSE: 0.00B [00:00, ?B/s]

...

You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.


Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2025.12.9: Fast Deepseekocr patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.
You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.


Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Some weights of DeepseekOCRForCausalLM were not initialized from the model checkpoint at ./deepseek_ocr and are newly initialized: ['model.vision_model.embeddings.position_ids']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


# Evaluate Deepseek-OCR Baseline Performance on Vietnamese Dataset

In [3]:
import zipfile

path_to_zip = "/content/UIT_HWDB_word.zip"
directory_to_extract = "/content/"

with zipfile.ZipFile(path_to_zip, 'r') as zip_ref:
    zip_ref.extractall(directory_to_extract)
    print("Đã giải nén thành công!")


Đã giải nén thành công!


In [12]:
import zipfile

path_to_zip = "/content/UIT_HWDB_line.zip"
directory_to_extract = "/content/"

with zipfile.ZipFile(path_to_zip, 'r') as zip_ref:
    zip_ref.extractall(directory_to_extract)
    print("Đã giải nén thành công!")


Đã giải nén thành công!


In [5]:
import zipfile

path_to_zip = "/content/crop_img.zip"
directory_to_extract = "/content/"

with zipfile.ZipFile(path_to_zip, 'r') as zip_ref:
    zip_ref.extractall(directory_to_extract)
    print("Đã giải nén thành công!")


Đã giải nén thành công!


# 1. Test thử với mô hình gốc

In [7]:
import jiwer
prompt = "<image>\nFree OCR. "
image_file = '/content/2.jpg'
output_path = 'your/output/dir'

# Tiny: base_size = 512, image_size = 512, crop_mode = False
# Small: base_size = 640, image_size = 640, crop_mode = False
# Base: base_size = 1024, image_size = 1024, crop_mode = False
# Large: base_size = 1280, image_size = 1280, crop_mode = False

# Gundam: base_size = 1024, image_size = 640, crop_mode = True

res = model.infer(tokenizer, prompt=prompt, image_file=image_file,
    output_path = output_path,
    image_size=640,
    base_size=1024,
    crop_mode=True,
    save_results = True,
    test_compress = False)
with open("/content/your/output/dir/result.mmd", "r") as f:
    predict_label = f.read()

true_label = "những người nói lên sự thật \" phát hành, nhiều bạn đọc đã bày tỏ sự chia sẻ,"

error = jiwer.cer(true_label, predict_label)
print(f"Character Error Rate (CER): {error}")

BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
having regard not to that 'that's him', there'd been a lot of big boys this set,
===============save results:===============


image: 0it [00:00, ?it/s]
other: 0it [00:00, ?it/s]

Character Error Rate (CER): 0.6842105263157895


# 2. Preprocessing

# 2.1 Preprocessing **word**

In [8]:
import json
data = []
for i in range(2, 52):
    with open('/content/UIT_HWDB_word/train_data/' + str(i) + '/label.json', 'r') as file:
    # Use json.load() to parse the file content into a Python object
        data.append(json.load(file))

print(data)


[{'1.jpg': 'Cả', '2.jpg': 'sới', '3.jpg': 'gà', '4.jpg': 'náo', '5.jpg': 'động', '6.jpg': 'Lượng', '7.jpg': 'tiền', '8.jpg': 'cốp', '9.jpg': 'tăng', '10.jpg': 'chóng', '11.jpg': 'mặt', '12.jpg': 'tỉ', '13.jpg': 'lệ', '14.jpg': 'cá', '15.jpg': 'cược', '16.jpg': 'thay', '17.jpg': 'đổi', '18.jpg': 'liên', '19.jpg': 'tục', '20.jpg': 'Chỉ', '21.jpg': 'trong', '22.jpg': 'ít', '23.jpg': 'phút', '24.jpg': 'lượng', '25.jpg': 'tiền', '26.jpg': 'cược', '27.jpg': 'trên', '28.jpg': 'sàn', '29.jpg': 'đã', '30.jpg': 'lên', '31.jpg': 'tới', '32.jpg': 'vài', '33.jpg': 'chục', '34.jpg': 'triệu', '35.jpg': 'đồng', '36.jpg': 'Đã', '37.jpg': 'hết', '38.jpg': 'hồ', '39.jpg': 'thứ', '40.jpg': 'sáu', '41.jpg': 'mỗi', '42.jpg': 'trận', '43.jpg': 'được', '44.jpg': 'tính', '45.jpg': 'làm', '46.jpg': '10', '47.jpg': 'hồ', '48.jpg': 'hiệp', '49.jpg': 'đấu', '50.jpg': 'mỗi', '51.jpg': 'hồ', '52.jpg': '15', '53.jpg': 'phút', '54.jpg': 'cặp', '55.jpg': 'gà', '56.jpg': 'vẫn', '57.jpg': 'chưa', '58.jpg': 'phân', '59.jp

In [9]:
train_data = {"/content/UIT_HWDB_word/train_data/2/1.jpg": data[0]['1.jpg']}
print(train_data)
from pathlib import Path
import os
path = Path("/content/UIT_HWDB_word/train_data")
# Duyệt qua các file (không bao gồm thư mục con sâu hơn)
for folder in path.iterdir():
    if folder.is_dir() and int(folder.name) >= 2 and int(folder.name) <= 51:
        path = Path("/content/UIT_HWDB_word/train_data/" + folder.name + "/")
        for file in folder.iterdir():
            if file.is_file():
                #print(file.name)
                if file.name != 'label.json':
                    full_path = os.path.join(path, file.name)
                    train_data[full_path] = data[int(folder.name) - 2][file.name]

print(train_data)
print(len(train_data))


{'/content/UIT_HWDB_word/train_data/2/1.jpg': 'Cả'}
{'/content/UIT_HWDB_word/train_data/2/1.jpg': 'Cả', '/content/UIT_HWDB_word/train_data/31/308.jpg': 'liệu', '/content/UIT_HWDB_word/train_data/31/132.jpg': 'món', '/content/UIT_HWDB_word/train_data/31/102.jpg': 'Thành', '/content/UIT_HWDB_word/train_data/31/294.jpg': 'hoàn', '/content/UIT_HWDB_word/train_data/31/218.jpg': 'giờ', '/content/UIT_HWDB_word/train_data/31/397.jpg': 'nguyên', '/content/UIT_HWDB_word/train_data/31/29.jpg': 'là', '/content/UIT_HWDB_word/train_data/31/214.jpg': 'ở', '/content/UIT_HWDB_word/train_data/31/398.jpg': 'của', '/content/UIT_HWDB_word/train_data/31/250.jpg': 'bị', '/content/UIT_HWDB_word/train_data/31/313.jpg': 'được', '/content/UIT_HWDB_word/train_data/31/378.jpg': 'NV3', '/content/UIT_HWDB_word/train_data/31/222.jpg': 'thường', '/content/UIT_HWDB_word/train_data/31/22.jpg': 'đời', '/content/UIT_HWDB_word/train_data/31/82.jpg': 'sống', '/content/UIT_HWDB_word/train_data/31/111.jpg': 'dũng', '/content/

In [10]:
instruction = "<image>\nFree OCR. "
from datasets import load_dataset
def convert_to_conversation(sample):
    """Convert dataset sample to conversation format"""
    conversation = [
        {
            "role": "<|User|>",
            "content": instruction,
            "images": [sample['image']]
        },
        {
            "role": "<|Assistant|>",
            "content": sample["text"]
        },
    ]
    return {"messages": conversation}



# 2.2 Preprocessing **line**

In [13]:

data_line = []
for i in range(52, 82):
    with open('/content/UIT_HWDB_line/train_data/' + str(i) + '/label.json', 'r') as file:
    # Use json.load() to parse the file content into a Python object
        data_line.append(json.load(file))

from pathlib import Path
import os
path = Path("/content/UIT_HWDB_line/train_data")
# Duyệt qua các file (không bao gồm thư mục con sâu hơn)
for folder in path.iterdir():
    if folder.is_dir() and int(folder.name) >= 52 and int(folder.name) <= 81:
        path = Path("/content/UIT_HWDB_line/train_data/" + folder.name + "/")
        for file in folder.iterdir():
            if file.is_file():
                #print(file.name)
                if file.name != 'label.json':
                    full_path = os.path.join(path, file.name)
                    train_data[full_path] = data_line[int(folder.name) - 52][file.name]
print(len(train_data))



23102


# 2.3 Preprocessing **line of colected dataset**

In [14]:
import json
import os

with open("/content/crop_img/rec_gt.txt", "r") as file:
    lines = file.readlines()

new_labels = {}
for line in lines:
    line = line.strip() # Remove leading/trailing whitespace, including newline
    if line:
        # Split on the first tab character to separate path and label
        parts = line.split('\t', 1)
        if len(parts) == 2:
            image_path = parts[0]
            label = parts[1]
            # Remove "crop_img/" prefix from the image path
            if image_path.startswith("crop_img/"):
                image_path = image_path[len("crop_img/"):]
            new_labels[image_path] = label

# Define the output directory and filename
output_dir = "/content/crop_img"
output_file = os.path.join(output_dir, "label.json")

# Ensure the output directory exists (though it should already)
os.makedirs(output_dir, exist_ok=True)

# Save the dictionary to a JSON file
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(new_labels, f, ensure_ascii=False, indent=4)

print(f"Created {output_file} successfully!")
# print(json.dumps(new_labels, ensure_ascii=False, indent=4))

Created /content/crop_img/label.json successfully!


In [15]:

data_line_cust = []
with open('/content/crop_img/' + "label.json", 'r') as file:
    # Use json.load() to parse the file content into a Python object
    data_line_cust.append(json.load(file))

path = Path("/content/crop_img")
# Duyệt qua các file (không bao gồm thư mục con sâu hơn)
for file in path.iterdir():
    if file.is_file():
                #print(file.name)
        if file.name != 'label.json' and file.name != 'rec_gt.txt':
            full_path = os.path.join(path, file.name)
            train_data[full_path] = data_line_cust[0][file.name]
print(len(train_data))



23907


In [16]:
from datasets import Dataset
from PIL import Image

dataset_items = []

for image_path, text_label in train_data.items():
    try:
        image = Image.open(image_path).convert("RGB")
        dataset_items.append({"image": image, "text": text_label})
    except FileNotFoundError:
        print(f"Warning: Image file not found at {image_path}. Skipping.")
    except Exception as e:
        print(f"Error processing {image_path}: {e}. Skipping.")

custom_dataset = Dataset.from_list(dataset_items)

print("Custom dataset created successfully!")
converted_dataset = [convert_to_conversation(sample) for sample in custom_dataset]
print("Dataset converted to conversation format successfully!")
print(f"Number of converted samples: {len(converted_dataset)}")
print("First converted sample:")
print(converted_dataset[-1])

Custom dataset created successfully!
Dataset converted to conversation format successfully!
Number of converted samples: 23907
First converted sample:
{'messages': [{'role': '<|User|>', 'content': '<image>\nFree OCR. ', 'images': [<PIL.PngImagePlugin.PngImageFile image mode=RGB size=36x32 at 0x7C33CAEF4CE0>]}, {'role': '<|Assistant|>', 'content': '13'}]}


# 3. Fine-tune model with custom_dataset

# Task
fine-tune

In [17]:
model = FastVisionModel.get_peft_model(
    model,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],

    r = 12,           # The larger, the higher the accuracy, but might overfit
    lora_alpha = 12,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
    # target_modules = "all-linear", # Optional now! Can specify a list if needed
)

Unsloth: Making `model.base_model.model.model` require gradients


# Create datacollator

In [18]:
# @title Create datacollator

import torch
import math
from dataclasses import dataclass
from typing import Dict, List, Any, Tuple
from PIL import Image, ImageOps
from torch.nn.utils.rnn import pad_sequence
import io

from deepseek_ocr.modeling_deepseekocr import (
    format_messages,
    text_encode,
    BasicImageTransform,
    dynamic_preprocess,
)

@dataclass
class DeepSeekOCRDataCollator:
    """
    Args:
        tokenizer: Tokenizer
        model: Model
        image_size: Size for image patches (default: 640)
        base_size: Size for global view (default: 1024)
        crop_mode: Whether to use dynamic cropping for large images
        train_on_responses_only: If True, only train on assistant responses (mask user prompts)
    """
    tokenizer: Any
    model: Any
    image_size: int = 640
    base_size: int = 1024
    crop_mode: bool = True
    image_token_id: int = 128815
    train_on_responses_only: bool = True

    def __init__(
        self,
        tokenizer,
        model,
        image_size: int = 640,
        base_size: int = 1024,
        crop_mode: bool = True,
        train_on_responses_only: bool = True,
    ):
        self.tokenizer = tokenizer
        self.model = model
        self.image_size = image_size
        self.base_size = base_size
        self.crop_mode = crop_mode
        self.image_token_id = 128815
        self.dtype = model.dtype  # Get dtype from model
        self.train_on_responses_only = train_on_responses_only

        self.image_transform = BasicImageTransform(
            mean=(0.5, 0.5, 0.5),
            std=(0.5, 0.5, 0.5),
            normalize=True
        )
        self.patch_size = 16
        self.downsample_ratio = 4

        # Get BOS token ID from tokenizer
        if hasattr(tokenizer, 'bos_token_id') and tokenizer.bos_token_id is not None:
            self.bos_id = tokenizer.bos_token_id
        else:
            self.bos_id = 0
            print(f"Warning: tokenizer has no bos_token_id, using default: {self.bos_id}")

    def deserialize_image(self, image_data) -> Image.Image:
        """Convert image data (bytes dict or PIL Image) to PIL Image in RGB mode"""
        if isinstance(image_data, Image.Image):
            return image_data.convert("RGB")
        elif isinstance(image_data, dict) and 'bytes' in image_data:
            image_bytes = image_data['bytes']
            image = Image.open(io.BytesIO(image_bytes))
            return image.convert("RGB")
        else:
            raise ValueError(f"Unsupported image format: {type(image_data)}")

    def calculate_image_token_count(self, image: Image.Image, crop_ratio: Tuple[int, int]) -> int:
        """Calculate the number of tokens this image will generate"""
        num_queries = math.ceil((self.image_size // self.patch_size) / self.downsample_ratio)
        num_queries_base = math.ceil((self.base_size // self.patch_size) / self.downsample_ratio)

        width_crop_num, height_crop_num = crop_ratio

        if self.crop_mode:
            img_tokens = num_queries_base * num_queries_base + 1
            if width_crop_num > 1 or height_crop_num > 1:
                img_tokens += (num_queries * width_crop_num + 1) * (num_queries * height_crop_num)
        else:
            img_tokens = num_queries * num_queries + 1

        return img_tokens

    def process_image(self, image: Image.Image) -> Tuple[List, List, List, List, Tuple[int, int]]:
        """
        Process a single image based on crop_mode and size thresholds

        Returns:
            Tuple of (images_list, images_crop_list, images_spatial_crop, tokenized_image, crop_ratio)
        """
        images_list = []
        images_crop_list = []
        images_spatial_crop = []

        if self.crop_mode:
            # Determine crop ratio based on image size
            if image.size[0] <= 640 and image.size[1] <= 640:
                crop_ratio = (1, 1)
                images_crop_raw = []
            else:
                images_crop_raw, crop_ratio = dynamic_preprocess(
                    image, min_num=2, max_num=9,
                    image_size=self.image_size, use_thumbnail=False
                )

            # Process global view with padding
            global_view = ImageOps.pad(
                image, (self.base_size, self.base_size),
                color=tuple(int(x * 255) for x in self.image_transform.mean)
            )
            images_list.append(self.image_transform(global_view).to(self.dtype))

            width_crop_num, height_crop_num = crop_ratio
            images_spatial_crop.append([width_crop_num, height_crop_num])

            # Process local views (crops) if applicable
            if width_crop_num > 1 or height_crop_num > 1:
                for crop_img in images_crop_raw:
                    images_crop_list.append(
                        self.image_transform(crop_img).to(self.dtype)
                    )

            # Calculate image tokens
            num_queries = math.ceil((self.image_size // self.patch_size) / self.downsample_ratio)
            num_queries_base = math.ceil((self.base_size // self.patch_size) / self.downsample_ratio)

            tokenized_image = ([self.image_token_id] * num_queries_base + [self.image_token_id]) * num_queries_base
            tokenized_image += [self.image_token_id]

            if width_crop_num > 1 or height_crop_num > 1:
                tokenized_image += ([self.image_token_id] * (num_queries * width_crop_num) + [self.image_token_id]) * (
                    num_queries * height_crop_num)

        else:  # crop_mode = False
            crop_ratio = (1, 1)
            images_spatial_crop.append([1, 1])

            # For smaller base sizes, resize; for larger, pad
            if self.base_size <= 640:
                resized_image = image.resize((self.base_size, self.base_size), Image.LANCZOS)
                images_list.append(self.image_transform(resized_image).to(self.dtype))
            else:
                global_view = ImageOps.pad(
                    image, (self.base_size, self.base_size),
                    color=tuple(int(x * 255) for x in self.image_transform.mean)
                )
                images_list.append(self.image_transform(global_view).to(self.dtype))

            num_queries = math.ceil((self.base_size // self.patch_size) / self.downsample_ratio)
            tokenized_image = ([self.image_token_id] * num_queries + [self.image_token_id]) * num_queries
            tokenized_image += [self.image_token_id]

        return images_list, images_crop_list, images_spatial_crop, tokenized_image, crop_ratio

    def process_single_sample(self, messages: List[Dict]) -> Dict[str, Any]:
            """
            Process a single conversation into model inputs.
            """

            # --- 1. Setup ---
            images = []
            for message in messages:
                if "images" in message and message["images"]:
                    for img_data in message["images"]:
                        if img_data is not None:
                            pil_image = self.deserialize_image(img_data)
                            images.append(pil_image)

            if not images:
                raise ValueError("No images found in sample. Please ensure all samples contain images.")

            tokenized_str = []
            images_seq_mask = []
            images_list, images_crop_list, images_spatial_crop = [], [], []

            prompt_token_count = -1 # Index to start training
            assistant_started = False
            image_idx = 0

            # Add BOS token at the very beginning
            tokenized_str.append(self.bos_id)
            images_seq_mask.append(False)

            for message in messages:
                role = message["role"]
                content = message["content"]

                # Check if this is the assistant's turn
                if role == "<|Assistant|>":
                    if not assistant_started:
                        # This is the split point. All tokens added *so far*
                        # are part of the prompt.
                        prompt_token_count = len(tokenized_str)
                        assistant_started = True

                    # Append the EOS token string to the *end* of assistant content
                    content = f"{content.strip()} {self.tokenizer.eos_token}"

                # Split this message's content by the image token
                text_splits = content.split('<image>')

                for i, text_sep in enumerate(text_splits):
                    # Tokenize the text part
                    tokenized_sep = text_encode(self.tokenizer, text_sep, bos=False, eos=False)
                    tokenized_str.extend(tokenized_sep)
                    images_seq_mask.extend([False] * len(tokenized_sep))

                    # If this text is followed by an <image> tag
                    if i < len(text_splits) - 1:
                        if image_idx >= len(images):
                            raise ValueError(
                                f"Data mismatch: Found '<image>' token but no corresponding image."
                            )

                        # Process the image
                        image = images[image_idx]
                        img_list, crop_list, spatial_crop, tok_img, _ = self.process_image(image)

                        images_list.extend(img_list)
                        images_crop_list.extend(crop_list)
                        images_spatial_crop.extend(spatial_crop)

                        # Add image placeholder tokens
                        tokenized_str.extend(tok_img)
                        images_seq_mask.extend([True] * len(tok_img))

                        image_idx += 1 # Move to the next image

            # --- 3. Validation and Final Prep ---
            if image_idx != len(images):
                raise ValueError(
                    f"Data mismatch: Found {len(images)} images but only {image_idx} '<image>' tokens were used."
                )

            # If we never found an assistant message, we're in a weird state
            # (e.g., user-only prompt). We mask everything.
            if not assistant_started:
                print("Warning: No assistant message found in sample. Masking all tokens.")
                prompt_token_count = len(tokenized_str)

            # Prepare image tensors
            images_ori = torch.stack(images_list, dim=0)
            images_spatial_crop_tensor = torch.tensor(images_spatial_crop, dtype=torch.long)

            if images_crop_list:
                images_crop = torch.stack(images_crop_list, dim=0)
            else:
                images_crop = torch.zeros((1, 3, self.base_size, self.base_size), dtype=self.dtype)

            return {
                "input_ids": torch.tensor(tokenized_str, dtype=torch.long),
                "images_seq_mask": torch.tensor(images_seq_mask, dtype=torch.bool),
                "images_ori": images_ori,
                "images_crop": images_crop,
                "images_spatial_crop": images_spatial_crop_tensor,
                "prompt_token_count": prompt_token_count, # This is now accurate
            }

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        """Collate batch of samples"""
        batch_data = []

        # Process each sample
        for feature in features:
            try:
                processed = self.process_single_sample(feature['messages'])
                batch_data.append(processed)
            except Exception as e:
                print(f"Error processing sample: {e}")
                continue

        if not batch_data:
            raise ValueError("No valid samples in batch")

        # Extract lists
        input_ids_list = [item['input_ids'] for item in batch_data]
        images_seq_mask_list = [item['images_seq_mask'] for item in batch_data]
        prompt_token_counts = [item['prompt_token_count'] for item in batch_data]

        # Pad sequences
        input_ids = pad_sequence(input_ids_list, batch_first=True, padding_value=self.tokenizer.pad_token_id)
        images_seq_mask = pad_sequence(images_seq_mask_list, batch_first=True, padding_value=False)

        # Create labels
        labels = input_ids.clone()

        # Mask padding tokens
        labels[labels == self.tokenizer.pad_token_id] = -100

        # Mask image tokens (model shouldn't predict these)
        labels[images_seq_mask] = -100

        # Mask user prompt tokens when train_on_responses_only=True (only train on assistant responses)
        if self.train_on_responses_only:
            for idx, prompt_count in enumerate(prompt_token_counts):
                if prompt_count > 0:
                    labels[idx, :prompt_count] = -100

        # Create attention mask
        attention_mask = (input_ids != self.tokenizer.pad_token_id).long()

        # Prepare images batch (list of tuples)
        images_batch = []
        for item in batch_data:
            images_batch.append((item['images_crop'], item['images_ori']))

        # Stack spatial crop info
        images_spatial_crop = torch.cat([item['images_spatial_crop'] for item in batch_data], dim=0)

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
            "images": images_batch,
            "images_seq_mask": images_seq_mask,
            "images_spatial_crop": images_spatial_crop,
        }

In [19]:
from transformers import Trainer, TrainingArguments
from unsloth import is_bf16_supported
FastVisionModel.for_training(model) # Enable for training!
data_collator = DeepSeekOCRDataCollator(
    tokenizer=tokenizer,
    model = model,
    image_size=640,
    base_size=1024,
    crop_mode=True,
    train_on_responses_only=True,
)
trainer = Trainer(
    model = model,
    tokenizer = tokenizer,
    data_collator = data_collator, # Must use!
    train_dataset = converted_dataset,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 50,
        # num_train_epochs = 1, # Set this instead of max_steps for full training runs
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        fp16 = not is_bf16_supported(),  # Use fp16 if bf16 is not supported
        bf16 = is_bf16_supported(),  # Use bf16 if supported
        output_dir = "outputs",
        report_to = "none",     # For Weights and Biases
        dataloader_num_workers=2,
        # You MUST put the below items for vision finetuning:
        remove_unused_columns = False,
    ),
)

/tmp/ipython-input-2133140209.py:12: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer._unsloth___init__`. Use `processing_class` instead.
  trainer = Trainer(


In [20]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 23,907 | Num Epochs = 1 | Total steps = 50
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 58,132,224 of 3,394,238,464 (1.71% trained)
Unsloth: Not an error, but DeepseekOCRForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


Step,Training Loss
1,7.618700
2,7.165900
3,4.542000
4,4.507500
5,5.288100
6,4.202400
7,3.287300
8,2.438200
9,2.814900
10,1.981000


BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
Unsloth: Will smartly offload gradients to save VRAM!
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([6, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 2

You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.


**Kiểm thư trên data trước đó**

In [21]:

prompt = "<image>\nFree OCR. "
image_file = '/content/2.jpg'
output_path = 'your/output/dir'

# Tiny: base_size = 512, image_size = 512, crop_mode = False
# Small: base_size = 640, image_size = 640, crop_mode = False
# Base: base_size = 1024, image_size = 1024, crop_mode = False
# Large: base_size = 1280, image_size = 1280, crop_mode = False

# Gundam: base_size = 1024, image_size = 640, crop_mode = True

res = model.infer(tokenizer, prompt=prompt, image_file=image_file,
    output_path = output_path,
    image_size=640,
    base_size=1024,
    crop_mode=True,
    save_results = True,
    test_compress = False)
with open("/content/your/output/dir/result.mmd", "r") as f:
    predict_label = f.read()

true_label = "những người nói lên sự thật \" phát hành, nhiều bạn đọc đã bày tỏ sự chia sẻ,"

error = jiwer.cer(true_label, predict_label)
print(f"Character Error Rate (CER): {error}")

BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
những người mới lên sự thật "thật hình, nhé! làm đọc dễ bấy ơi, chưa 
===============save results:===============


image: 0it [00:00, ?it/s]
other: 0it [00:00, ?it/s]

Character Error Rate (CER): 0.32894736842105265


# Kết luận
Như vậy, CER đã tăng đáng kể (từ 0.68 xuống còn 0.33 với file 2.png), một kết quả khả quan.

# 4. Save model

In [22]:
model.save_pretrained("lora_model")  # Local saving
tokenizer.save_pretrained("lora_model")
# model.push_to_hub("your_name/lora_model", token = "...") # Online saving
# tokenizer.push_to_hub("your_name/lora_model", token = "...") # Online saving

You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.


('lora_model/tokenizer_config.json',
 'lora_model/special_tokens_map.json',
 'lora_model/tokenizer.json')

In [23]:
if False:
    from unsloth import FastVisionModel
    model, tokenizer = FastVisionModel.from_pretrained(
        model_name = "lora_model", # YOUR MODEL YOU USED FOR TRAINING
        load_in_4bit = False, # Use 4bit to reduce memory use. False for 16bit LoRA.
        auto_model = AutoModel,
        trust_remote_code=True,
        unsloth_force_compile=True,
        use_gradient_checkpointing = "unsloth", # True or "unsloth" for long context
    )
    FastVisionModel.for_inference(model) # Enable for inference!

prompt = "<image>\nFree OCR. "
image_file = '/content/2.jpg'
output_path = 'your/output/dir'



res = model.infer(tokenizer, prompt=prompt, image_file=image_file,
    output_path = output_path,
    image_size=640,
    base_size=1024,
    crop_mode=True,
    save_results = True,
    test_compress = False)


BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
những người mới lên sự thật "thật hình, nhé! làm đọc dễ bấy ơi, chưa 
===============save results:===============


image: 0it [00:00, ?it/s]
other: 0it [00:00, ?it/s]


# 5. Đánh giá CER trên mô hình gốc và fine-tune

In [25]:
import os

print("Listing contents of /content/UIT_HWDB_line/test_data/")
print(os.listdir('/content/UIT_HWDB_line/test_data/'))

Listing contents of /content/UIT_HWDB_line/test_data/
['253', '255', '252', '250', '251', '254']


In [28]:
import json
from pathlib import Path
import os
from PIL import Image
from datasets import Dataset

# Define the actual test data folders found in /content/UIT_HWDB_line/test_data/
actual_test_folders = ['250', '251', '252', '253', '254', '255']

# 1. Initialize an empty dictionary called `test_data` to store image paths and their labels.
test_data = {}

# 2. Iterate through the actual test folders to populate `test_data`
for folder_name in actual_test_folders:
    folder_path = Path(f"/content/UIT_HWDB_line/test_data/{folder_name}/")

    if folder_path.is_dir():
        label_file_path = folder_path / 'label.json'
        try:
            with open(label_file_path, 'r') as file:
                folder_labels = json.load(file)

            for file_name in folder_path.iterdir():
                if file_name.is_file() and file_name.name != 'label.json':
                    full_path = str(file_name)
                    try:
                        label = folder_labels[file_name.name]
                        test_data[full_path] = label
                    except KeyError:
                        print(f"Error: Label for {file_name.name} not found in {label_file_path}. Skipping.")
        except FileNotFoundError:
            print(f"Warning: label.json not found in {folder_path}. Skipping this folder.")
        except Exception as e:
            print(f"Error loading label.json from {folder_path}: {e}. Skipping this folder.")
    else:
        print(f"Warning: Folder {folder_path} not found. Skipping.")

print(f"Loaded {len(test_data)} test image-label pairs.")

# 3. Create an empty list called `test_dataset_items`
test_dataset_items = []

# 4. Iterate through `test_data` to open images and append to `test_dataset_items`
for image_path, text_label in test_data.items():
    try:
        image = Image.open(image_path).convert("RGB")
        test_dataset_items.append({"image": image, "text": text_label})
    except FileNotFoundError:
        print(f"Warning: Image file not found at {image_path}. Skipping.")
    except Exception as e:
        print(f"Error processing {image_path}: {e}. Skipping.")

# 5. Convert `test_dataset_items` into a `Dataset` object
custom_test_dataset = Dataset.from_list(test_dataset_items)

print("Custom test dataset created successfully!")

# 6. Apply the `convert_to_conversation` function (defined in a previous cell)
converted_test_dataset = [convert_to_conversation(sample) for sample in custom_test_dataset]

print("Test dataset converted to conversation format successfully!")
print(f"Number of converted test samples: {len(converted_test_dataset)}")
print("First converted test sample:")
print(converted_test_dataset[0] if converted_test_dataset else "No samples.")


Loaded 201 test image-label pairs.
Custom test dataset created successfully!
Test dataset converted to conversation format successfully!
Number of converted test samples: 201
First converted test sample:
{'messages': [{'role': '<|User|>', 'content': '<image>\nFree OCR. ', 'images': [<PIL.PngImagePlugin.PngImageFile image mode=RGB size=2048x190 at 0x7C33C595EB40>]}, {'role': '<|Assistant|>', 'content': 'quyền thu hồi đất, giao đất phải được thể hiện theo tinh thần " ông bố bao trùm'}]}


## Define CER evaluation function




In [29]:
import jiwer
import os
import shutil

def evaluate_cer(model, tokenizer, dataset):
    """
    Calculates the average Character Error Rate (CER) on a given dataset
    using a specified model and tokenizer.

    Args:
        model: The vision model to use for inference.
        tokenizer: The tokenizer associated with the model.
        dataset: The dataset containing images and true labels.

    Returns:
        float: The average CER across the dataset.
    """
    all_cers = []
    # Create a temporary directory for inference output
    temp_output_dir = "temp_inference_results"
    os.makedirs(temp_output_dir, exist_ok=True)

    # Use the same prompt as used during training
    inference_prompt = "<image>\nFree OCR. "

    print(f"Starting CER evaluation on {len(dataset)} samples...")

    for i, sample in enumerate(dataset):
        if (i + 1) % 50 == 0: # Print progress every 50 samples
            print(f"Processing sample {i + 1}/{len(dataset)}")

        try:
            # Extract image and true label
            image = sample['messages'][0]['images'][0] # PIL Image object
            true_label = sample['messages'][1]['content']

            # Save the PIL image to a temporary file for model.infer
            # DeepseekOCR model.infer expects a file path
            temp_image_path = os.path.join(temp_output_dir, f"temp_image_{i}.png")
            image.save(temp_image_path)

            # Perform inference
            # The model.infer function automatically saves results to result.mmd in output_path
            model.infer(tokenizer, prompt=inference_prompt, image_file=temp_image_path,
                        output_path=temp_output_dir,
                        image_size=640,
                        base_size=1024,
                        crop_mode=True,
                        save_results=True,
                        test_compress=False)

            # Read the predicted label from the saved file
            result_file_path = os.path.join(temp_output_dir, "result.mmd")
            if os.path.exists(result_file_path):
                with open(result_file_path, "r", encoding="utf-8") as f:
                    predicted_label = f.read().strip()

                # Calculate CER and append
                cer_value = jiwer.cer(true_label, predicted_label)
                all_cers.append(cer_value)
            else:
                print(f"Warning: result.mmd not found for sample {i}. Skipping CER calculation for this sample.")

            # Clean up temporary image file
            os.remove(temp_image_path)

        except Exception as e:
            print(f"Error processing sample {i}: {e}. Skipping.")

    # Clean up the temporary directory after evaluation
    shutil.rmtree(temp_output_dir)
    print("Finished CER evaluation.")

    if not all_cers:
        print("No CER values were calculated.")
        return 1.0 # Or raise an error, or return 0.0 depending on desired behavior

    average_cer = sum(all_cers) / len(all_cers)
    return average_cer

print("CER evaluation function defined.")

CER evaluation function defined.


## Evaluate original model




In [30]:
print("Reloading original Deepseek-OCR model...")
original_model, original_tokenizer = FastVisionModel.from_pretrained(
    "./deepseek_ocr",
    load_in_4bit = False,
    auto_model = AutoModel,
    trust_remote_code=True,
    unsloth_force_compile=True,
    use_gradient_checkpointing = "unsloth",
)

print("Evaluating original model CER...")
original_model_cer = evaluate_cer(original_model, original_tokenizer, converted_test_dataset)
print(f"Original Model Average CER: {original_model_cer}")

You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.


Reloading original Deepseek-OCR model...
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2025.12.9: Fast Deepseekocr patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.
You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.


Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Some weights of DeepseekOCRForCausalLM were not initialized from the model checkpoint at ./deepseek_ocr and are newly initialized: ['model.vision_model.embeddings.position_ids']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
mai ... đen chảy. Những nhìn màn còn, mẹ vẫn còn cười. Ôi vì hai bố con không thể thành
===============save results:===============


image: 0it [00:00, ?it/s]
other: 0it [00:00, ?it/s]


BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([8, 100, 1280])
Thanh công ấy, loài cỏ mây người đạt được ?
===============save results:===============


image: 0it [00:00, ?it/s]
other: 0it [00:00, ?it/s]

Finished CER evaluation.
Original Model Average CER: 0.3255953521567368


In [ ]:
if True:
    from unsloth import FastVisionModel
    model, tokenizer = FastVisionModel.from_pretrained(
        model_name = "lora_model", # YOUR MODEL YOU USED FOR TRAINING
        load_in_4bit = False, # Use 4bit to reduce memory use. False for 16bit LoRA.
        auto_model = AutoModel,
        trust_remote_code=True,
        unsloth_force_compile=True,
        use_gradient_checkpointing = "unsloth", # True or "unsloth" for long context
    )
    FastVisionModel.for_inference(model)
print("Evaluating fine-tune model CER...")
finetune_model_cer = evaluate_cer(model, tokenizer, converted_test_dataset)
print(f"Fine-tune Model Average CER: {fine-tune_model_cer}")